# Practice 4. Finding the right clause: diagnose and fix

**ML in Industry 2026 · Topic 4: Retrieval-augmented generation**

## Goal

The engineer asks again: *"After a bearing change, pump P-101 vibrates at 4.8 mm/s. Is this normal?"*
The answer is in regulation R-12, but the plant archive now has 16 documents: two revisions of R-12 and an
amendment, a near-copy of R-12 for another pump station, a maintenance regulation, a CMMS procedure, datasheets,
an English manual of an imported pump, a valve list with 110 rows and some unrelated instructions.

Search mechanisms (embeddings, hybrid search, rerankers) are taught in the Advanced NLP course. Here search is a
ready module, and we do the engineering around it:

1. a **retrieval set** built from the structure of the documents, with known evidence for every question;
2. measure the baseline and **attribute every lost answer** to a link: reading / search / context assembly / model;
3. **fix one link at a time** with plant knowledge, each fix is a row of an **ablation table** with a paired
   comparison (practice 2);
4. mechanisms from Advanced NLP are added as **rows of the same table**;
5. the answer follows a **contract**; code checks the citations; "not in the documents" and "conflict" are answers.

```
question ─► fields (tags, date) ─► registry: station, class ─► filter: scope, date ─► BM25 ─► evidence pack
                                                                                               │
                         audit log ◄── 3 citation checks ◄── answer contract (course Qwen) ◄───┘
```

## Steps and what you should see

| Step | What we do | Expected result |
|------|------------|-----------------|
| 0 | The final table | the ablation table S0-S6 and the mechanism rows |
| 1 | Setup | the course model answers; otherwise replay |
| 2 | Archive v2 | 16 documents, 440 chunks, new fields, `index_version` |
| 3 | Retrieval set | 143 questions in 12 slices, gold chunks computed by code; a holdout of 27 written by role |
| 4 | Baseline S0 | about half of the answers right; most losses in search |
| 5 | Ladder S1-S5 | tags, table rows (mixed or separate), applicability, dates, evidence pack: one row each |
| 6 | Mechanisms | a reranker and query translation as rows of the same table, with their cost |
| 7 | Answer and checks | P-101 answered with checked citations; a neighbour clause caught; conflict; abstention |
| 8 | Audit, regression, holdout | an audit record; the regression gate S6 vs S5; the ladder on questions written by role |
| 9 | Reproduce | the mandatory path, success criteria, optional tasks |

## Live or replay

Search, filters and evidence packs run **live** (milliseconds, no model). Model answers for all 143 questions
in every configuration come from the **replay** of the lecturer's run: identical requests get identical saved
answers. Single questions in steps 5 and 7 go to the course model live if it answers. Dense vectors and
reranker answers are replayed too (when present).

## Common problems

| Problem | What to do |
|---------|-----------|
| `Mode: REPLAY` but you have the course token | check `COURSE_LLM_BASE_URL` and `COURSE_LLM_API_KEY` in `.env`; run `uv run python -m plant_assistant.smoke_test course` |
| `ReplayMiss` | you changed a prompt, the chunking, a filter or the questions: replay works only for unchanged requests. Run live, or undo the change |
| your table differs from `reference/ablation.csv` in retrieval columns | the store differs: compare `index_version` with `reference/store/manifest.json` |
| the answer is empty, `finish_reason = length` | thinking mode is on: keep `course` (thinking off) for the answer contract |

## 0. Where we are going

The final table of this practice, measured by the lecturer. At the end you will know where every number comes from.
Every row adds one change to the row in `vs` and is compared with it **in pairs, on the same questions**.

In [1]:
import warnings

import pandas as pd
from IPython.display import Markdown, display

from plant_assistant import REFERENCE_DIR

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 250)
reference = pd.read_csv(REFERENCE_DIR / "ablation.csv").fillna("")
reference[["config", "vs", "recall@5", "evidence recall", "end-to-end", "correct + verified", "e2e verdict"]]

,config,vs,recall@5,evidence recall,end-to-end,correct + verified,e2e verdict
0,S0,,61/123 = 50% [41%–58%],54/123 = 44% [35%–53%],73/143 = 51% [43%–59%],47/143 = 33% [26%–41%],
1,S1,S0,65/123 = 53% [44%–61%],58/123 = 47% [39%–56%],78/143 = 55% [46%–62%],51/143 = 36% [28%–44%],no detectable difference (p = 0.062)
2,S2,S1,69/123 = 56% [47%–65%],59/123 = 48% [39%–57%],77/143 = 54% [46%–62%],52/143 = 36% [29%–45%],no detectable difference (p = 1.000)
3,S2b,S1,67/123 = 54% [46%–63%],60/123 = 49% [40%–58%],80/143 = 56% [48%–64%],51/143 = 36% [28%–44%],no detectable difference (p = 0.625)
4,S3,S2b,82/123 = 67% [58%–74%],84/123 = 68% [60%–76%],102/143 = 71% [63%–78%],62/143 = 43% [36%–52%],B is better (p = 0.000)
5,S4,S3,93/123 = 76% [67%–82%],94/123 = 76% [68%–83%],109/143 = 76% [69%–82%],109/143 = 76% [69%–82%],B is better (p = 0.016)
6,S5,S4,93/123 = 76% [67%–82%],108/123 = 88% [81%–92%],123/143 = 86% [79%–91%],123/143 = 86% [79%–91%],B is better (p = 0.000)
7,S6,S5,93/123 = 76% [67%–82%],108/123 = 88% [81%–92%],130/143 = 91% [85%–95%],128/143 = 90% [83%–94%],B is better (p = 0.039)
8,M1t,S0,9/18 = 50% [29%–71%],9/18 = 50% [29%–71%],13/20 = 65% [43%–82%],5/20 = 25% [11%–47%],B is better (p = 0.021)
9,M3@S0,S0,93/123 = 76% [67%–82%],90/123 = 73% [65%–80%],101/143 = 71% [63%–77%],58/143 = 41% [33%–49%],B is better (p = 0.000)


## 1. Setup

Profiles are in `models.toml`, secrets in `.env` (as in practices 1-3). The answers use the course model
Qwen 3.8 27B with thinking off. Embedding profiles (`embed-*`) are optional: on 27.10.2026 the course server has
no embedding or reranking model, so dense vectors come from `data/replay/vectors/` when the lecturer adds them.

In [2]:
import os

from plant_assistant import REPLAY_DIR, RUNS_DIR, STORE_DIR
from plant_assistant.config import get_profile, list_profiles, load_env
from plant_assistant.llm import CallLogger, LLMClient, ReplayClient

load_env()
PROFILE = os.environ.get("PRACTICE_PROFILE", "course")
pd.DataFrame(list_profiles().items(), columns=["profile", "description"])

,profile,description
0,course,"Course model Qwen 3.8 27B via LiteLLM + vLLM, thinking off"
1,course-thinking,"Course model Qwen 3.8 27B, thinking on (spends tokens before the JSON: raise max_tokens)"
2,local,"Small open-weight model on your laptop via Ollama (Qwen3.5 4B, Q4)"
3,embed-bge-m3,Multilingual embeddings bge-m3 (567M) via Ollama: ollama pull bge-m3
4,embed-qwen3,Multilingual embeddings Qwen3-Embedding 0.6B via Ollama: ollama pull qwen3-embedding:0.6b
5,embed-course,"Embeddings on the course server, if the lab deploys a model (variant A of the design)"
6,drafter,Model that drafts holdout questions: gpt-oss-120b on the course server (a frontier profile is better)


In [3]:
def model_answers(profile_name: str) -> bool:
    # One call without network retries: if the model is not there, we want to know now.
    try:
        client = LLMClient(get_profile(profile_name), max_network_retries=0)
    except ValueError as exc:  # profile not configured (no URL or key)
        print(f"{profile_name}: {exc}")
        return False
    r = client.chat([{"role": "user", "content": "Reply with one word: OK"}])
    print(f"{profile_name}: {r.content.strip()[:20]!r} ({r.latency_s:.1f}s)" if r.ok else r.error[:200])
    return r.ok


LIVE = os.environ.get("PRACTICE_MODE") != "replay" and model_answers(PROFILE)
replay = {task: ReplayClient(REPLAY_DIR / f"{task}_course.jsonl") for task in ("answers", "rerank", "translate")}
live = LLMClient(get_profile(PROFILE), logger=CallLogger(RUNS_DIR / f"my-{PROFILE}" / "calls.jsonl")) if LIVE \
    else replay["answers"]
print("Mode:", f"LIVE single questions on {PROFILE}, the full set from replay" if LIVE else
      "REPLAY of the lecturer's saved answers (course model)")
vectors = sorted((REPLAY_DIR / "vectors").glob("*.npz"))
print("Embedding vectors:", [v.stem for v in vectors] or "none yet: dense rows (M1, M2) are not measured")

Mode: REPLAY of the lecturer's saved answers (course model)
Embedding vectors: none yet: dense rows (M1, M2) are not measured


## 2. The plant archive, version 2

The store of practice 3 (R-12 rev. 2 and rev. 3, two datasheets) plus 12 new **synthetic** documents. They are
born-digital: reading was topic 3 and must not get in the way. This is a teaching scale (tens of pages, hundreds
of chunks), not "thousands of pages": the process is the same, the numbers are illustrative.

`build_corpus` reads the new PDFs, takes dates and scope from the document registry (`data/documents.csv`, as from
a document management system), adds the new fields and writes `store/`.

In [4]:
from plant_assistant.corpus import build_corpus, load_registry
from plant_assistant.store import load_store

summary = build_corpus(STORE_DIR)
print(summary)
_, chunks = load_store(STORE_DIR)
by_id = {c.chunk_id: c for c in chunks}
docs = pd.DataFrame(load_registry())
docs[["doc_id", "revision", "amendment", "kind", "language", "effective_from", "effective_to", "scope_stations",
      "scope_equipment_types", "scope_tags", "pages", "source"]]

{'documents': 16, 'pages': 47, 'chunks': 440, 'chunks_without_rows': 330, 'table_rows': 110, 'index_version': 'corpus-5719c6dc76.parser-a53367.emb-none', 'seconds': 4.9}


,doc_id,revision,amendment,kind,language,effective_from,effective_to,scope_stations,scope_equipment_types,scope_tags,pages,source
0,DS-P-205,NaN,NaN,datasheet,zh,2023-01-01,None,[2],[pump],[P-205],1,practice 3
1,PS-101,NaN,NaN,datasheet,ru,2019-01-01,None,[2],[pump],[P-101],1,practice 3
2,R-12,2.0,NaN,regulation,ru,2022-06-01,2025-03-14,[2],"[pump, compressor, fan]",[],6,practice 3
3,R-12,3.0,NaN,regulation,ru,2025-03-14,None,[2],"[pump, compressor, fan]",[],6,practice 3
4,FS-07,2.0,NaN,instruction,ru,2024-01-15,None,[2],[all],[],3,practice 4
5,LAB-03,1.0,NaN,procedure,ru,2022-03-01,None,[lab],[all],[],2,practice 4
6,OM-P-205,2.0,NaN,manual,en,2023-05-15,None,[2],[pump],[P-205],3,practice 4
7,PS-102,NaN,NaN,datasheet,ru,2018-01-01,None,[2],[pump],[P-102],1,practice 4
8,PS-310,NaN,NaN,datasheet,ru,2021-01-01,None,[3],[pump],[P-310],1,practice 4
9,PS-311,NaN,NaN,datasheet,ru,2021-01-01,None,[3],[pump],[P-311],1,practice 4


**What is new, and why.** An **amendment** to R-12 rev. 3 changes one clause (4.4.1: 720 h → 500 h from
01.09.2026). **R-14** is a near-copy of R-12 for pump station No. 3, with the same title and clause numbers and a
stricter Table 3. The **manual of P-205** is in English and gives milder limits than R-12: a conflict.
The **valve list** has a table of 110 rows over four pages. R-20 defines what "priority high" in clause 6.2 of
R-12 means.

> **Chat:** "R-12 and R-14 have the same title. Name one field that tells them apart."

In [5]:
from plant_assistant.display import where

for cid in ("R-12-r3-table1", "R-14-r1-table1"):
    c = by_id[cid]
    print(f"{where(c)}  scope_stations={c.scope_stations}  valid {c.effective_from} → {c.effective_to or 'now'}")
    print("   ", c.text.splitlines()[0], "|", [line for line in c.text.splitlines() if "|II|" in line][0])

R-12 rev. 3 Table 3  scope_stations=['2']  valid 2025-03-14 → now
    Таблица 3 — Границы виброскорости по классам машин (СКЗ, мм/с) | |Насосы|II|4,5|7,1|
R-14 rev. 1 Table 3  scope_stations=['3']  valid 2024-03-01 → now
    Таблица 3 — Границы виброскорости по классам машин (СКЗ, мм/с) | |Насосы|II|3,5|5,6|


New fields of a chunk (the practice 3 schema is kept, the fields are added):
`effective_from` / `effective_to` (validity), `amended_by`, `scope_stations` / `scope_equipment_types` /
`scope_tags`, `references` / `referenced_by` (links in both directions), `table_no`, `row_key`, `doc_language`.

In [6]:
fields = ["chunk_id", "clause", "table_no", "row_key", "effective_from", "effective_to", "amended_by", "references",
          "referenced_by"]
pd.DataFrame([{f: getattr(by_id[cid], f) for f in fields} for cid in
              ("R-12-r3-4.4.1", "R-12-r3-a1-4.4.1", "R-12-r3-4.3.3", "R-12-r3-6.2", "R-20-r2-3.3",
               "VL-2-r4-tА.2-V-12")])

,chunk_id,clause,table_no,row_key,effective_from,effective_to,amended_by,references,referenced_by
0,R-12-r3-4.4.1,4.4.1,None,None,2025-03-14,2026-09-01,R-12-r3-a1-4.4.1,[],[R-12-r3-a1-4.4.1]
1,R-12-r3-a1-4.4.1,4.4.1,None,None,2026-09-01,None,None,[R-12-r3-4.4.1],[]
2,R-12-r3-4.3.3,4.3.3,None,None,2025-03-14,None,None,[R-12-r3-table1],[R-12-r3-6.2]
3,R-12-r3-6.2,6.2,None,None,2025-03-14,None,None,"[R-12-r3-4.3.3, R-12-r3-4.3.4]",[R-20-r2-3.3]
4,R-20-r2-3.3,3.3,None,None,2023-01-10,None,None,"[R-12-r2-6.2, R-12-r3-6.2, R-14-r1-6.2]",[]
5,VL-2-r4-tА.2-V-12,А,А.2,V-12,2025-11-20,None,None,[],[]


The index has a version: a fingerprint of the chunks, the parser and the embedding model. It goes into every
answer log (step 8). If any of the three changes, the index is rebuilt and the retrieval set runs again (topic 6).

In [7]:
print("index_version:", summary["index_version"])
pd.crosstab(pd.Series([c.doc_id for c in chunks], name="document"), pd.Series([c.chunk_type for c in chunks],
                                                                              name="chunk type"), margins=True)

index_version: corpus-5719c6dc76.parser-a53367.emb-none


chunk type,clause,datasheet,note,section,table,table_row,All
document,,,,,,,
DS-P-205,0,1,0,0,0,0,1
FS-07,16,0,0,5,1,0,22
LAB-03,12,0,0,4,0,0,16
OM-P-205,21,0,0,7,3,0,31
PS-101,0,1,0,0,0,0,1
PS-102,0,1,0,0,0,0,1
PS-310,0,1,0,0,0,0,1
PS-311,0,1,0,0,0,0,1
R-12,86,0,2,28,6,0,122


## 3. The retrieval set

143 questions in the format of the practice 2 harness. The 50 questions of practice 3 (known page) are the
`base` slice; 20 of them also go to search **in English as they are** (the `language` slice); the rest are new.
Every question has a Russian wording (it goes to search and to the prompt: the engineer of a Russian plant asks in
Russian) and an English translation for us. Evidence is written as **locators**; the gold chunk ids are computed
by code. Slices are reported as **counts**: 3 of 4 is not "75%".

In [8]:
from plant_assistant.cases import counts_by_slice, load_cases

cases = load_cases()
case_by_id = {c.case_id: c for c in cases}
pd.DataFrame([counts_by_slice(cases)], index=["questions"])

,base,language,applicability,point_in_time,incident,what_changed,multi_evidence,cross_doc,conflict,table_row,tag_variant,not_in_documents
questions,50,20,12,10,3,4,8,3,9,8,6,10


In [9]:
from plant_assistant.display import gold_frame

P101 = case_by_id["m01"]
print("Question (asked in Russian):", P101.question_ru)
print("Translation:                ", P101.question_en)
print("Expected:", P101.expected, "| asked on", P101.asked_on)
gold_frame(P101)

Question (asked in Russian): После замены подшипника насос P-101 вибрирует 4,8 мм/с. Это норма? Что делать и в какой срок?
Translation:                 After a bearing change pump P-101 vibrates at 4.8 mm/s. Is this normal? What must be done, and by when?
Expected: {'values': ['4,5', '24'], 'unit': 'h'} | asked on 2026-10-27


,role,locator (any of),gold chunk ids
0,answer,R-12 r3 cl 4.3.3,R-12-r3-4.3.3
1,support,R-12 r3 t 3,R-12-r3-table1
2,action,R-12 r3 cl 6.1,R-12-r3-6.1
3,action,R-12 r3 cl 6.2,R-12-r3-6.2


Four evidence chunks for one answer: the clause with the deadline (4.3.3), the table with the limit (Table 3),
and the actions (6.1, 6.2). recall@k counts only the answer; **evidence recall** asks for all four in the context.

## 4. Baseline S0: BM25 over raw chunks, top-5, no filters

> **Predict, then run.** Where will most answers be lost: reading, search, context assembly or the model?

Search runs live; the answers of the course model for 143 questions come from replay.

In [10]:
from plant_assistant import reval
from plant_assistant.display import answer_frame, context_frame, hits_frame
from plant_assistant.pipeline import CONFIGS, LADDER, System

system = System(chunks, answer_client=replay["answers"], rerank_client=replay["rerank"],
                translate_client=replay["translate"])
records, scores = {}, {}


def run(name: str) -> pd.DataFrame:
    records[name] = system.run(cases, CONFIGS[name], progress=False)
    scores[name] = reval.score(cases, records[name], by_id)
    return scores[name]


run("S0")
reval.ablation({"S0": scores["S0"]}, {})[["config", "recall@5", "evidence recall", "MRR", "end-to-end",
                                          "correct + verified", "abstain ok", "conflict ok"]]

,config,recall@5,evidence recall,MRR,end-to-end,correct + verified,abstain ok,conflict ok
0,S0,61/123 = 50% [41%–58%],54/123 = 44% [35%–53%],0.247,73/143 = 51% [43%–59%],47/143 = 33% [26%–41%],10/10,0/9


In [11]:
display(Markdown("**Wrong answers of S0 by link**"))
pd.DataFrame([reval.losses(scores["S0"])], index=["S0"])

**Wrong answers of S0 by link**

,reading,search,assembly,model
S0,0,60,2,8


Most answers are lost **in search**: the chunk with the answer never reached the model. Nothing is lost in
reading (the documents are born-digital and the practice 3 store was read well). In practice 3 the model answered
98% of these questions when it was given the page: everything below that is lost by search and assembly here.

The same view for the P-101 question: the top-5 of BM25.

In [12]:
rec = next(r for r in records["S0"] if r["case_id"] == "m01")
hits_frame(rec, P101, by_id)

,chunk,valid,gold,text
rank,,,,
1,R-12 rev. 2 cl. 4.4.2,2022-06-01 → 2025-03-14,,"4.4.2 После ремонта, замены подшипников или центровки измерения выполняют в течение 8 часов после пуска."
2,R-12 rev. 3 cl. 4.4.2,2025-03-14 → now,,"4.4.2 После ремонта, замены подшипников или центровки измерения выполняют в течение 8 часов после пуска."
3,R-14 rev. 1 cl. 4.4.2,2024-03-01 → now,,"4.4.2 После ремонта, замены подшипников или центровки измерения выполняют в течение 8 часов после пуска."
4,R-15 rev. 1 cl. 4.3,2023-09-01 → now,,4.3 Марку смазочного материала берут из паспорта насоса. Замена марки без согласования с главным механиком не
5,R-15 rev. 1 cl. 5.1,2023-09-01 → now,,5.1 Центровку валов насоса и двигателя проверяют при каждом ТО-1 и после каждого ремонта.


No word of the question is in clause 4.3.3 ("при превышении границы осмотра ... осматривают в течение 24 часов"):
lexical search cannot find it from "вибрирует 4,8 мм/с, это норма?". Only Table 3 comes up, by "мм/с".

**Task 1 of the lecture** used five failures of exactly this run:

In [13]:
display(Markdown((REFERENCE_DIR / "s0_failures.md").read_text(encoding="utf-8")))

# Five failed answers of S0 (task 1 of lecture 4)

Configuration S0: BM25 over raw chunks, top-5, no filters, the course model answers from the top-5.
Task: for each answer write where it was lost: R = reading, S = search, C = context assembly, M = model.

## 1. m01 (multi_evidence) — lost in: **search**

- Question (asked in Russian): После замены подшипника насос P-101 вибрирует 4,8 мм/с. Это норма? Что делать и в какой срок?
- Translation: After a bearing change pump P-101 vibrates at 4.8 mm/s. Is this normal? What must be done, and by when?
- Expected: ['4,5', '24'] h
- System answer: The sources do not specify the vibration limit to determine if 4.8 mm/s is normal, but they mandate that measurements be taken within 8 hours after startup following a bearing replacement. You must perform the measurement within this 8-hour window. (value: None)
- Gold evidence: R-12 r3 cl 4.3.3; R-12 r3 t 3; R-12 r3 cl 6.1; R-12 r3 cl 6.2
- Top-5 of search: R-12 rev. 2 cl. 4.4.2; R-12 rev. 3 cl. 4.4.2; R-14 rev. 1 cl. 4.4.2; R-15 rev. 1 cl. 4.3; R-15 rev. 1 cl. 5.1
- Why this link: no chunk with the answer reached the context

## 2. g01 (tag_variant) — lost in: **search**

- Question (asked in Russian): Какая мощность двигателя у насоса Р-101?
- Translation: What is the motor power of pump Р-101 (Cyrillic Р)?
- Expected: 55 kW
- System answer: The provided sources do not contain the specific power rating of the motor for pump R-101. (value: None)
- Gold evidence: PS-101
- Top-5 of search: R-12 rev. 2 cl. 4.5.1; R-12 rev. 3 cl. 4.5.1; R-14 rev. 1 cl. 4.5.1; R-15 rev. 1 cl. 5.1; R-12 rev. 2 cl. 7.2
- Why this link: no chunk with the answer reached the context

## 3. v03 (point_in_time) — lost in: **search**

- Question (asked in Russian): С какой периодичностью сейчас нужно контролировать вибрацию насоса P-101 переносным виброметром?
- Translation: How often must the vibration of pump P-101 be checked with a portable meter now?
- Expected: 500 h
- System answer: Vibration of pump P-101 must be controlled with a portable vibrometer every 1000 operating hours, but not less than once a month. (value: 1000)
- Gold evidence: R-12 r3a1 cl 4.4.1
- Top-5 of search: R-12 rev. 2 cl. 3.7; R-12 rev. 3 cl. 3.7; R-14 rev. 1 cl. 3.7; R-12 rev. 2 cl. 4.2.1; R-12 rev. 2 cl. 4.4.1
- Why this link: no chunk with the answer reached the context

## 4. m06 (multi_evidence) — lost in: **assembly**

- Question (asked in Russian): После замены подшипника насос P-101 запустили. В течение скольких часов нужно измерить вибрацию и с какой границей сравнить результат?
- Translation: Pump P-101 was started after a bearing change. Within how many hours must the vibration be measured, and against which limit?
- Expected: ['8', '4,5'] h
- System answer: Vibration measurements must be taken within 8 hours after startup. The sources do not specify the comparison limit. (value: 8)
- Gold evidence: R-12 r3 cl 4.4.2; R-12 r3 t 3
- Top-5 of search: R-12 rev. 2 cl. 4.4.2; R-12 rev. 3 cl. 4.4.2; R-14 rev. 1 cl. 4.4.2; FS-07 rev. 2 cl. 3.3; R-12 rev. 2 cl. 6.1
- Why this link: the chunk with the answer is in the context, a required support or action chunk is not

## 5. i01 (incident) — lost in: **model**

- Question (asked in Russian): 10.02.2025 насос P-101 вибрировал 4,8 мм/с, осмотр в течение суток не провели. Было ли это нарушением регламента?
- Translation: On 10 February 2025 pump P-101 vibrated at 4.8 mm/s and nobody inspected it within 24 hours. Was this a violation of the regulation?
- Expected: ['5,0'] mm/s
- System answer: Yes, it was a violation of the regulation. The vibration of 4.8 mm/s exceeded the inspection threshold of 4.5 mm/s for a Class II pump, requiring an inspection within 24 hours, which was not performed. (value: 4,5)
- Gold evidence: R-12 r2 t 3; R-12 r2 cl 4.3.3
- Top-5 of search: R-12 rev. 2 cl. 4.3.3; R-12 rev. 3 cl. 4.3.3; R-14 rev. 1 cl. 4.3.3; R-12 rev. 2 Table 3; R-12 rev. 3 Table 3
- Why this link: every needed chunk was in the context, the answer is still wrong


## 5. The ladder: one fix per row

Every row adds one change to the previous one and is compared with it in pairs. We look at one question before and
after, then at the row of the table.

**S1: tag normalization.** "Р-101" with a Cyrillic Р, "P101", "P 102": one form in the chunks **and** in the query.

In [14]:
for name in ("S0", "S1"):
    rec = system.run_case(case_by_id["g02"], CONFIGS[name], with_answer=False)
    display(Markdown(f"**{name}** · {case_by_id['g02'].question_ru}"), hits_frame(rec, case_by_id["g02"], by_id))

**S0** · Какая подача у насоса P101?

,chunk,valid,gold,text
rank,,,,
1,R-15 rev. 1 cl. 4.1,2023-09-01 → now,,"4.1 Подшипники качения насосов смазывают каждые 3000 часов наработки смазкой, указанной в паспорте насоса."
2,R-14 rev. 1 Table А.1,2024-03-01 → now,,"Таблица А.1 — Оборудование насосной станции № 3 |Тег|Наименование|Мощность<br>,<br>кВт|Класс|Паспорт| |---|---"
3,FS-07 rev. 2 cl. 2.2,2024-01-15 → now,,2.2 Проходы к насосам и электрическим щитам шириной не менее 1 м содержат свободными.
4,R-15 rev. 1 cl. 5.1,2023-09-01 → now,,5.1 Центровку валов насоса и двигателя проверяют при каждом ТО-1 и после каждого ремонта.
5,R-12 rev. 2 Table А.1,2022-06-01 → 2025-03-14,,"Таблица А.1 — Оборудование насосной станции № 2 |Тег|Наименование|Мощность<br>,<br>кВт|Класс|Паспорт| |---|---"


**S1** · Какая подача у насоса P101?

,chunk,valid,gold,text
rank,,,,
1,R-12 rev. 2 Table А.1,2022-06-01 → 2025-03-14,,"Таблица А.1 — Оборудование насосной станции № 2 |Тег|Наименование|Мощность<br>,<br>кВт|Класс|Паспорт| |---|---"
2,R-12 rev. 3 Table А.1,2025-03-14 → now,,"Таблица А.1 — Оборудование насосной станции № 2 |Тег|Наименование|Мощность<br>,<br>кВт|Класс|Паспорт| |---|---"
3,FS-07 rev. 2 Table 1,2024-01-15 → now,,Таблица 1 — Нормы оснащения помещения насосной станции № 2 |Средство|Количество|Место| |---|---|---| |Огнетуши
4,R-15 rev. 1 cl. 4.1,2023-09-01 → now,,"4.1 Подшипники качения насосов смазывают каждые 3000 часов наработки смазкой, указанной в паспорте насоса."
5,PS-101,2019-01-01 → now,answer,# ПАСПОРТ НАСОСА # PUMP DATASHEET <table> <tr> <th>Тег / Tag</th> <td>P-101</td> </tr> <tr> <th>Изго


**S2: rows of long tables are chunks.** The valve list has 110 rows on 4 pages. A page of 33 rows is one chunk in
S0-S1; in S2 every row is a chunk with the caption, the column names and the units:

In [15]:
print(by_id["VL-2-r4-tА.2-V-12"].text)
for name in ("S1", "S2"):
    rec = system.run_case(case_by_id["r01"], CONFIGS[name], with_answer=False)
    display(Markdown(f"**{name}** · {case_by_id['r01'].question_ru}"), hits_frame(rec, case_by_id["r01"], by_id, 3))

Таблица А.2 — Перечень запорной арматуры насосной станции № 2. Строка V-12. Тег: V-12; Тип: ЗД; DN, мм: 150; PN, МПа: 1,6; Среда: нефть; Трубопровод: нагнетание P-101; Привод: ручной


**S1** · Какое допустимое давление для задвижки V-12?

,chunk,valid,gold,text
rank,,,,
1,VL-2 rev. 4 cl. 1.3,2025-11-20 → now,support,"1.3 Арматуру эксплуатируют при рабочем давлении, не превышающем номинальное давление PN, указанное в таблице А"
2,R-15 rev. 1 cl. 6.1,2023-09-01 → now,,6.1 Допустимая утечка через торцевое уплотнение — не более 10 капель в минуту.
3,R-15 rev. 1 cl. 6.2,2023-09-01 → now,,6.2 При превышении допустимой утечки начальник смены создаёт заявку с приоритетом «средний» по R-20.


**S2** · Какое допустимое давление для задвижки V-12?

,chunk,valid,gold,text
rank,,,,
1,VL-2 rev. 4 cl. 1.3,2025-11-20 → now,support,"1.3 Арматуру эксплуатируют при рабочем давлении, не превышающем номинальное давление PN, указанное в таблице А"
2,VL-2 rev. 4 Table А.2 row V-12,2025-11-20 → now,answer,"Таблица А.2 — Перечень запорной арматуры насосной станции № 2. Строка V-12. Тег: V-12; Тип: ЗД; DN, мм: 150; P"
3,R-15 rev. 1 cl. 6.1,2023-09-01 → now,,6.1 Допустимая утечка через торцевое уплотнение — не более 10 капель в минуту.


**S2b: rows in a separate collection.** 110 short row chunks mixed with the clauses win BM25 for every question
that names a pump (the rows say "нагнетание P-101"), and push datasheets out of the top-5. Instead: rows are
searched in their own index and get a small quota (top-5 chunks + at most 2 rows that share a term with the
question). The questions that mixed rows broke in the lecturer's first run:

In [16]:
from plant_assistant.pipeline import PREVIOUS

previous = dict(PREVIOUS)
broken = ["d01", "d02", "d04", "z01", "z02", "z04", "z05", "i02", "g01"]
for name in ("S1", "S2", "S2b"):
    run(name)
pd.DataFrame({name: scores[name].set_index("case_id").loc[broken, "correct"] for name in ("S1", "S2", "S2b")}).T

case_id,d01,d02,d04,z01,z02,z04,z05,i02,g01
S1,False,False,True,True,True,True,True,True,True
S2,False,False,False,False,False,False,False,False,False
S2b,False,False,True,True,True,True,True,True,True


**S3: applicability through the registry.** The tag goes to the equipment registry: station, type, class and the
**documents of the unit** (datasheet, manufacturer's manual). The filter keeps only documents whose scope fits,
**before** search; the class goes into the prompt; the tables of the linked documents go into the context. P-310
is at station No. 3, so R-14 applies, not R-12. The manual of P-205 does not name the plant tag (the manufacturer
does not know it): only the registry links it.

In [17]:
from plant_assistant.applicability import load_registry

print(load_registry()["P-205"].describe())
rec = system.run_case(case_by_id["f01"], CONFIGS["S3"], with_answer=False)
print("linked by the registry:", [where(by_id[c]) for c in rec["linked"]])

P-205: pump, Pump station 2, motor 37 kW, machine class II, documents DS-P-205, OM-P-205
linked by the registry: ['DS-P-205', 'OM-P-205 rev. 2 Table 1', 'OM-P-205 rev. 2 Table 2', 'OM-P-205 rev. 2 Table 3']


In [18]:
from plant_assistant.applicability import equipment_context

# single questions: live if the course model answers (the reranker too), replay otherwise
system_live = System(chunks, answer_client=live, rerank_client=live if LIVE else replay["rerank"],
                     translate_client=live if LIVE else replay["translate"])
for name in ("S2", "S3"):
    rec = system_live.run_case(case_by_id["p01"], CONFIGS[name])
    display(Markdown(f"**{name}** · {case_by_id['p01'].question_ru} · scope: {rec['scope'] or 'none'}"),
            hits_frame(rec, case_by_id["p01"], by_id, 3))
    print("answer:", (rec["answer"] or {}).get("answer"))

**S2** · Насос P-310 вибрирует 4,0 мм/с. Нужно ли его осматривать? · scope: none

,chunk,valid,gold,text
rank,,,,
1,R-14 rev. 1 Table А.1,2024-03-01 → now,support,"Таблица А.1 — Оборудование насосной станции № 3 |Тег|Наименование|Мощность<br>,<br>кВт|Класс|Паспорт| |---|---"
2,R-12 rev. 2 Table 3,2022-06-01 → 2025-03-14,,"Таблица 3 — Границы виброскорости по классам машин (СКЗ, мм/с) |Группа оборудования|Класс<br>машины|Граница ви"
3,R-12 rev. 3 Table 3,2025-03-14 → now,,"Таблица 3 — Границы виброскорости по классам машин (СКЗ, мм/с) |Группа оборудования|Класс<br>машины|Граница ви"


answer: Yes, pump P-310 requires inspection because its vibration of 4.0 mm/s exceeds the inspection threshold of 3.5 mm/s for Class II pumps. The pump should be inspected immediately.


**S3** · Насос P-310 вибрирует 4,0 мм/с. Нужно ли его осматривать? · scope: stations=['3'], types=['pump'], tags=['P-310']

,chunk,valid,gold,text
rank,,,,
1,R-14 rev. 1 Table А.1,2024-03-01 → now,support,"Таблица А.1 — Оборудование насосной станции № 3 |Тег|Наименование|Мощность<br>,<br>кВт|Класс|Паспорт| |---|---"
2,R-14 rev. 1 Table 3,2024-03-01 → now,answer,"Таблица 3 — Границы виброскорости по классам машин (СКЗ, мм/с) |Группа оборудования|Класс<br>машины|Граница ви"
3,R-14 rev. 1 cl. 3.3,2024-03-01 → now,,"3.3 Граница осмотра: значение виброскорости, при превышении которого оборудование осматривают в установленный"


answer: Yes, the pump P-310 must be inspected because its vibration of 4.0 mm/s exceeds the inspection limit of 3.5 mm/s for a Class II machine. The inspection must be performed within 24 hours, and measurements must be repeated every shift until the inspection is complete.


**S4: a revision filter by the date of the question.** `effective_from <= date < effective_to`, not
`status = in force`. The same question on three dates: 1000 h (rev. 2), 720 h (rev. 3), 500 h (amendment No. 1).
At S4 the old wordings are gone, but the amendment is not in the top-5 for "now" (its text differs from the
question): the model says "not in the documents". The evidence pack of S5 brings it as a part of section 4.4.

In [19]:
for cid in ("v01", "v02", "v03"):
    for name in ("S3", "S4", "S5"):
        rec = system_live.run_case(case_by_id[cid], CONFIGS[name])
        a = rec["answer"] or {}
        print(f"{cid} {name}: as_of {rec['as_of']}  expected {case_by_id[cid].expected['value']} h  "
              f"answer {a.get('value')} {a.get('unit')}  top-1 {where(by_id[rec['retrieved'][0]])}")

v01 S3: as_of 2024-05-01  expected 1000 h  answer 720 hours  top-1 R-12 rev. 2 section 4.4
v01 S4: as_of 2024-05-01  expected 1000 h  answer 1000 hours  top-1 R-12 rev. 2 section 4.4
v01 S5: as_of 2024-05-01  expected 1000 h  answer 1000 hours  top-1 R-12 rev. 2 section 4.4
v02 S3: as_of 2025-06-01  expected 720 h  answer 720 hours  top-1 R-12 rev. 2 section 4.4
v02 S4: as_of 2025-06-01  expected 720 h  answer 720 hours  top-1 R-12 rev. 3 section 4.4
v02 S5: as_of 2025-06-01  expected 720 h  answer 720 hours  top-1 R-12 rev. 3 section 4.4
v03 S3: as_of 2026-10-27  expected 500 h  answer 1000 hours  top-1 R-12 rev. 2 cl. 3.7
v03 S4: as_of 2026-10-27  expected 500 h  answer None None  top-1 R-12 rev. 3 cl. 3.7
v03 S5: as_of 2026-10-27  expected 500 h  answer 500 hours  top-1 R-12 rev. 3 cl. 3.7


**Typical mistakes, measured.** A filter *after* search empties the top-k; the flag `status = in force` answers a
question about an incident in February 2025 with rev. 3.

In [20]:
from plant_assistant.filters import make_filter, post_filter, status_in_force
from plant_assistant.query import extract_rules

q = case_by_id["i01"]
fields = extract_rules(q.query, q.asked_on)
idx = system.bm25(CONFIGS["S4"])
by_date = make_filter(fields, None, use_scope=False, use_date=True)
pre = [where(by_id[h.chunk_id]) for h in idx.search(q.query, 5, by_date)]
post = [where(by_id[h.chunk_id]) for h in post_filter(idx.search(q.query, 5), by_id, by_date)]
flag = [where(by_id[h.chunk_id]) for h in idx.search(q.query, 5, status_in_force)]
print("question:", q.question_en)
pd.DataFrame({"pre-filter by date": pd.Series(pre), "post-filter by date": pd.Series(post),
              "status = in force": pd.Series(flag)})

question: On 10 February 2025 pump P-101 vibrated at 4.8 mm/s and nobody inspected it within 24 hours. Was this a violation of the regulation?


,pre-filter by date,post-filter by date,status = in force
0,R-12 rev. 2 cl. 4.3.3,R-12 rev. 2 cl. 4.3.3,R-12 rev. 3 cl. 4.3.3
1,R-14 rev. 1 cl. 4.3.3,R-14 rev. 1 cl. 4.3.3,R-14 rev. 1 cl. 4.3.3
2,R-12 rev. 2 Table 3,R-12 rev. 2 Table 3,R-12 rev. 3 Table 3
3,R-14 rev. 1 Table 3,NaN,R-14 rev. 1 Table 3
4,R-12 rev. 2 cl. 6.1,NaN,R-12 rev. 3 cl. 6.1


**S5: the evidence pack.** For every hit: the chunks it refers to and the chunks that refer to it (6.2 refers to
4.3.3, not the other way round), one more step back, then the whole sections, within 2500 tokens.

In [21]:
for name in ("S4", "S5"):
    rec = system.run_case(P101, CONFIGS[name], with_answer=False)
    ctx = context_frame(rec, P101, by_id)
    print(f"{name}: {len(ctx)} chunks, {ctx.tokens.sum()} tokens; gold in the context: "
          f"{sorted(set(ctx.gold) - {''})}, evidence recall = {reval.retrieval_metrics(P101, rec)['evidence_recall']}")
ctx[ctx.gold != ""]

S4: 6 chunks, 760 tokens; gold in the context: [], evidence recall = False
S5: 18 chunks, 1047 tokens; gold in the context: [], evidence recall = False


,chunk,gold,tokens


> **Poll:** "Which fix will give the biggest gain on the whole set? Tag normalization / Table rows /
> Applicability / Revision filter / Evidence pack"

The rows of the table, each against the previous one:

In [22]:
for name in ("S3", "S4", "S5"):
    run(name)
cols = ["config", "vs", "recall@5", "evidence recall", "end-to-end", "e2e: fixed / broke", "e2e verdict",
        "evidence verdict"]
reval.ablation(scores, previous)[cols].fillna("")

,config,vs,recall@5,evidence recall,end-to-end,e2e: fixed / broke,e2e verdict,evidence verdict
0,S0,,61/123 = 50% [41%–58%],54/123 = 44% [35%–53%],73/143 = 51% [43%–59%],,,
1,S1,S0,65/123 = 53% [44%–61%],58/123 = 47% [39%–56%],78/143 = 55% [46%–62%],+5 / -0,no detectable difference (p = 0.062),"no detectable difference (+4/-0, p = 0.125)"
2,S2,S1,69/123 = 56% [47%–65%],59/123 = 48% [39%–57%],77/143 = 54% [46%–62%],+6 / -7,no detectable difference (p = 1.000),"no detectable difference (+4/-3, p = 1.000)"
3,S2b,S1,67/123 = 54% [46%–63%],60/123 = 49% [40%–58%],80/143 = 56% [48%–64%],+3 / -1,no detectable difference (p = 0.625),"no detectable difference (+2/-0, p = 0.500)"
4,S3,S2b,82/123 = 67% [58%–74%],84/123 = 68% [60%–76%],102/143 = 71% [63%–78%],+24 / -2,B is better (p = 0.000),"B is better (+24/-0, p = 0.000)"
5,S4,S3,93/123 = 76% [67%–82%],94/123 = 76% [68%–83%],109/143 = 76% [69%–82%],+7 / -0,B is better (p = 0.016),"B is better (+10/-0, p = 0.002)"
6,S5,S4,93/123 = 76% [67%–82%],108/123 = 88% [81%–92%],123/143 = 86% [79%–91%],+14 / -0,B is better (p = 0.000),"B is better (+14/-0, p = 0.000)"


In [23]:
display(Markdown("**End-to-end by slice** (counts)"))
reval.by_slice(scores, "correct")

**End-to-end by slice** (counts)

,base,language,applicability,point_in_time,incident,what_changed,multi_evidence,cross_doc,conflict,table_row,tag_variant,not_in_documents
config,,,,,,,,,,,,
S0,41/50,5/20,2/12,4/10,1/3,1/4,3/8,0/3,0/9,6/8,0/6,10/10
S1,41/50,5/20,2/12,4/10,1/3,1/4,3/8,0/3,0/9,6/8,5/6,10/10
S2,38/50,5/20,4/12,4/10,0/3,1/4,3/8,0/3,0/9,8/8,4/6,10/10
S2b,41/50,5/20,1/12,5/10,1/3,1/4,3/8,0/3,0/9,8/8,5/6,10/10
S3,49/50,10/20,8/12,6/10,1/3,1/4,3/8,0/3,0/9,8/8,6/6,10/10
S4,49/50,10/20,10/12,9/10,3/3,1/4,3/8,0/3,0/9,8/8,6/6,10/10
S5,50/50,11/20,12/12,10/10,3/3,3/4,7/8,3/3,0/9,8/8,6/6,10/10


Read the table by slices, not only by the total: S2 fixes table rows and **breaks** base questions, S2b keeps
the fix without the damage; S3 moves `applicability` and `base`, S4 `point_in_time` and `incident`, S5
`multi_evidence` and `cross_doc`. The `conflict` slice stays at 0 until S6: without the contract the model has no
way to say "the documents disagree".

## 6. Mechanisms from Advanced NLP: rows of the same table

Two points of attachment: **instead of** plant knowledge (on S0) and **on top of** it (on S6). We do not explain
how they work; we look at where they help.

- **M3** an LLM reranker (listwise, course Qwen) over the top-20 of the current search;
- **M1t** translation of an English question into Russian before search (the `language` slice only);
- **M1 / M2** dense and hybrid search: need embedding vectors (see below).

> **Chat:** "Will a reranker fix the point-in-time slice? yes / no"

In [24]:
for name in ("S6", "M1t", "M3@S0", "M3@S6"):
    run(name)
table = reval.ablation(scores, previous)
table[table.config.isin(["S0", "M1t", "M3@S0", "S6", "M3@S6"])][cols].fillna("")

,config,vs,recall@5,evidence recall,end-to-end,e2e: fixed / broke,e2e verdict,evidence verdict
0,S0,,61/123 = 50% [41%–58%],54/123 = 44% [35%–53%],73/143 = 51% [43%–59%],,,
7,S6,S5,93/123 = 76% [67%–82%],108/123 = 88% [81%–92%],130/143 = 91% [85%–95%],+8 / -1,B is better (p = 0.039),"no detectable difference (+0/-0, p = 1.000)"
8,M1t,S0,9/18 = 50% [29%–71%],9/18 = 50% [29%–71%],13/20 = 65% [43%–82%],+9 / -1,B is better (p = 0.021),"no detectable difference (+8/-2, p = 0.109)"
9,M3@S0,S0,93/123 = 76% [67%–82%],90/123 = 73% [65%–80%],101/143 = 71% [63%–77%],+32 / -4,B is better (p = 0.000),"B is better (+39/-3, p = 0.000)"
10,M3@S6,S6,104/123 = 85% [77%–90%],115/123 = 93% [88%–97%],130/143 = 91% [85%–95%],+5 / -5,no detectable difference (p = 1.000),"B is better (+7/-0, p = 0.016)"


In [25]:
reval.by_slice({k: scores[k] for k in ("S0", "M3@S0", "S6", "M3@S6")}, "correct")

,base,language,applicability,point_in_time,incident,what_changed,multi_evidence,cross_doc,conflict,table_row,tag_variant,not_in_documents
config,,,,,,,,,,,,
S0,41/50,5/20,2/12,4/10,1/3,1/4,3/8,0/3,0/9,6/8,0/6,10/10
M3@S0,47/50,10/20,8/12,7/10,2/3,3/4,3/8,1/3,0/9,8/8,2/6,10/10
S6,50/50,11/20,12/12,10/10,3/3,3/4,7/8,2/3,8/9,8/8,6/6,10/10
M3@S6,47/50,13/20,12/12,10/10,3/3,3/4,8/8,2/3,8/9,8/8,6/6,10/10


**What each row costs.** Median seconds per question (search here + model latency saved with the answers), model
calls and tokens per question. A row that is better must also be affordable: the reranker doubles the calls.

In [26]:
reval.cost_table(records)

,median s / question,p90 s,search ms (median),model calls / question,prompt tokens / question,completion tokens / question
config,,,,,,
S0,1.46,2.28,2.8,1.0,843.0,84.0
S1,1.46,2.26,0.3,1.0,864.0,84.0
S2,1.40,2.12,0.3,1.0,727.0,84.0
S2b,1.48,2.13,4.2,1.0,839.0,106.0
S3,1.68,2.25,0.5,1.0,1083.0,84.0
S4,1.77,2.60,0.5,1.0,1135.0,83.0
S5,3.05,4.06,0.7,1.0,2811.0,88.0
S6,4.77,8.90,0.7,1.0,3036.0,374.0
M1t,1.93,2.52,0.4,2.0,769.0,102.0


In [27]:
lang = scores["M1t"].case_id
display(Markdown("**The `language` slice**: English questions as they are (S0) vs translated to Russian (M1t)"))
pd.DataFrame({name: [f"{int(scores[name].set_index('case_id').loc[lang, m].astype(bool).sum())}/{len(lang)}"
                     for m in ("recall@k", "correct")] for name in ("S0", "M1t")}, index=["recall@5", "end-to-end"])

**The `language` slice**: English questions as they are (S0) vs translated to Russian (M1t)

,S0,M1t
recall@5,3/20,9/20
end-to-end,5/20,13/20


**Dense search (M1, M2).** On 27.10.2026 the course server has no embedding model, and the lecturer's vectors are
not in `data/replay/vectors/` yet. With Ollama you can make them yourself (optional task 2):
`ollama pull bge-m3`, then `uv run python scripts/build_vectors.py --profile embed-bge-m3 --name bge-m3`.
The cell below runs M1 and M2 when vectors are there.

In [28]:
from dataclasses import replace

from plant_assistant.search import VectorStore

if vectors:
    system.vectors = VectorStore(vectors[0])
    system.vectors.check(system.index_version)
    for name in ("M1@S0", "M2@S0", "M1@S6", "M2@S6"):
        run(name)
    previous.update({"M1@S0": "S0", "M2@S0": "S0", "M1@S6": "S6", "M2@S6": "S6"})
    display(reval.ablation(scores, previous)[cols].fillna(""))
else:
    print("No vectors: M1 and M2 are TODO in the reference table too.")

No vectors: M1 and M2 are TODO in the reference table too.


## 7. The answer contract and the citation checks

S6 = S5 + the answer contract: a status (`answered`, `not_in_documents`, `conflict`, `needs_human`), the value with
its unit, claims with their sources, and citations as fields that code can check:

1. **exists**: a chunk with this document, revision, clause / table / row is in the index;
2. **in force**: it is valid on the date of the question;
3. **supports**: the number and the unit of the answer are in its text (or in a table it refers to).

The P-101 question. With the archive v2, BM25 puts Table 3 at rank 6 for this wording (a lubrication clause of
R-15 shares "подшипник" and "насос"), so S6 has nothing to build the evidence pack from. The reranker over the
top-20 brings Table 3 back, and the pack does the rest:

In [29]:
for name in ("S6", "M3@S6"):
    rec = system_live.run_case(P101, CONFIGS[name])
    print(f"{name}: top-5 {[where(by_id[c]) for c in rec['top']]}")
    print(f"     status {rec['answer']['status']}, correct {reval.grade(P101, rec['answer'])[0]}, "
          f"verified {rec['checks']['verified']}")
answer, checks = answer_frame(rec)
display(answer)
checks

S6: top-5 ['R-12 rev. 3 cl. 4.4.2', 'R-15 rev. 1 cl. 4.3', 'R-15 rev. 1 cl. 5.1', 'R-15 rev. 1 cl. 4.1', 'FS-07 rev. 2 Table 1']
     status not_in_documents, correct False, verified True
M3@S6: top-5 ['R-12 rev. 3 Table А.1', 'R-12 rev. 3 Table 3', 'R-12 rev. 3 section 6', 'R-20 rev. 2 Table 1', 'R-12 rev. 3 cl. 4.4.2']
     status answered, correct True, verified True


,value
field,
status,answered
answer,The vibration of 4.8 mm/s exceeds the inspection limit (4.5 mm/s) for a Class II pump but is below the stop limit (7...
value,"4,5"
unit,мм/с


,source,chunk_id,exists,in_force,supports
0,S1,R-12-r3-table3,True,True,False
1,S2,R-12-r3-table1,True,True,True
2,S6,R-12-r3-4.3.3,True,True,True
3,S7,R-12-r3-6.2,True,True,False
4,S10,R-20-r2-table1,True,True,False
5,S14,PS-101-p1,True,True,False
6,S20,R-12-r3-6.1,True,True,False


> **Chat:** "The answer cites R-12 rev. 2. Which check fails?"

Task 3 of the lecture: *"Inspect P-101 within 24 hours (R-12, rev. 3, clause 4.3.4, page 3)"*. The clause exists
and is in force; its text is about the **stop** limit, "24 hours" is in the neighbouring clause 4.3.3.

In [30]:
from plant_assistant.answer import Answer, Citation
from plant_assistant.verify import check_answer

claim = Answer(status="answered", answer="Inspect P-101 within 24 hours.", value="24", unit="h",
               citations=[Citation(source="S1", doc_id="R-12", revision=3, clause="4.3.4", page=3)])
rows = []
for rev, clause in ((3, "4.3.4"), (3, "4.3.3"), (2, "4.3.3")):
    claim.citations[0].revision, claim.citations[0].clause = rev, clause
    r = check_answer(claim, [], chunks, by_id, P101.asked_on)
    rows.append({"citation": f"R-12 rev. {rev} cl. {clause}", "exists": r["exists"], "in force": r["in_force"],
                 "supports 24 h": r["supports"], "verified": r["verified"]})
pd.DataFrame(rows)

,citation,exists,in force,supports 24 h,verified
0,R-12 rev. 3 cl. 4.3.4,True,True,False,False
1,R-12 rev. 3 cl. 4.3.3,True,True,True,True
2,R-12 rev. 2 cl. 4.3.3,True,False,True,False


**Conflict.** P-205 is a class II pump at station No. 2: R-12 says inspect above 4.5 mm/s; the manufacturer's
manual says "alarm 7.1 mm/s". BM25 never finds the English manual from a Russian question, but the registry links
it (S3), so both sides are in the context. In the same language: R-15 (maintenance) says lubricate every 3000 h,
R-12 says 2000 h, both in force for station No. 2. The system must show both, not choose:

In [31]:
for cid in ("f01", "f05"):
    rec = system_live.run_case(case_by_id[cid], CONFIGS["S6"])
    cited = sorted({by_id[c["chunk_id"]].doc_id for c in rec["checks"]["citations"] if c["chunk_id"]})
    print(f"{cid}: {rec['answer']['status']} | cited {cited} | {rec['answer']['answer'][:220]}")

f01: conflict | cited ['OM-P-205', 'R-12'] | The plant regulation R-12 sets the inspection limit for a Class II pump at 4.5 mm/s, requiring inspection within 24 hours. The manufacturer's manual OM-P-205 sets the alarm limit at 7.1 mm/s, requiring the cause to be fo
f05: conflict | cited ['PS-101', 'R-12', 'R-15'] | The sources provide conflicting intervals for lubricating the rolling bearings of pump P-101. Document R-15 (clause 4.1) specifies lubrication every 3000 hours of operation. Document R-12 (clause 7.1) specifies lubricati


**Not in the documents.** V-912 is in no document (V-112 is in the registry but in no document either):

In [32]:
for cid in ("x01", "x03"):
    rec = system_live.run_case(case_by_id[cid], CONFIGS["S6"])
    print(cid, rec["answer"]["status"], "|", rec["answer"]["answer"][:160])

x01 needs_human | The sources do not contain the specific nominal pressure (PN) for valve V-912. Clause 1.3 of VL-2 states that the operating pressure must not exceed the nominal
x03 not_in_documents | The provided sources do not contain information about the allowable pressure for valve V-112. The sources only define the scope and terminology for the CMMS mai


The whole ladder with S6:

In [33]:
full = reval.ablation(scores, previous)
full[full.config.isin([c.name for c in LADDER])][cols + ["correct + verified", "abstain ok", "conflict ok"]].fillna("")

,config,vs,recall@5,evidence recall,end-to-end,e2e: fixed / broke,e2e verdict,evidence verdict,correct + verified,abstain ok,conflict ok
0,S0,,61/123 = 50% [41%–58%],54/123 = 44% [35%–53%],73/143 = 51% [43%–59%],,,,47/143 = 33% [26%–41%],10/10,0/9
1,S1,S0,65/123 = 53% [44%–61%],58/123 = 47% [39%–56%],78/143 = 55% [46%–62%],+5 / -0,no detectable difference (p = 0.062),"no detectable difference (+4/-0, p = 0.125)",51/143 = 36% [28%–44%],10/10,0/9
2,S2,S1,69/123 = 56% [47%–65%],59/123 = 48% [39%–57%],77/143 = 54% [46%–62%],+6 / -7,no detectable difference (p = 1.000),"no detectable difference (+4/-3, p = 1.000)",52/143 = 36% [29%–45%],10/10,0/9
3,S2b,S1,67/123 = 54% [46%–63%],60/123 = 49% [40%–58%],80/143 = 56% [48%–64%],+3 / -1,no detectable difference (p = 0.625),"no detectable difference (+2/-0, p = 0.500)",51/143 = 36% [28%–44%],10/10,0/9
4,S3,S2b,82/123 = 67% [58%–74%],84/123 = 68% [60%–76%],102/143 = 71% [63%–78%],+24 / -2,B is better (p = 0.000),"B is better (+24/-0, p = 0.000)",62/143 = 43% [36%–52%],10/10,0/9
5,S4,S3,93/123 = 76% [67%–82%],94/123 = 76% [68%–83%],109/143 = 76% [69%–82%],+7 / -0,B is better (p = 0.016),"B is better (+10/-0, p = 0.002)",109/143 = 76% [69%–82%],10/10,0/9
6,S5,S4,93/123 = 76% [67%–82%],108/123 = 88% [81%–92%],123/143 = 86% [79%–91%],+14 / -0,B is better (p = 0.000),"B is better (+14/-0, p = 0.000)",123/143 = 86% [79%–91%],10/10,0/9
7,S6,S5,93/123 = 76% [67%–82%],108/123 = 88% [81%–92%],130/143 = 91% [85%–95%],+8 / -1,B is better (p = 0.039),"no detectable difference (+0/-0, p = 1.000)",128/143 = 90% [83%–94%],10/10,8/9


## 8. The audit log and the regression gate

Half a year later, "why did the system say 5,0?" must have an answer. The record keeps the question, its date,
the index version, the chunks with their revisions, the answer and the checks.

In [34]:
import json

from plant_assistant.audit import audit_record

rec = next(r for r in records["M3@S6"] if r["case_id"] == "m01")
print(json.dumps(audit_record(rec, system.index_version, by_id), ensure_ascii=False, indent=1)[:1800])

{
 "ts": "2026-10-27T20:11:02+00:00",
 "case_id": "m01",
 "question": "После замены подшипника насос P-101 вибрирует 4,8 мм/с. Это норма? Что делать и в какой срок?",
 "asked_on": "2026-10-27",
 "as_of": "2026-10-27",
 "config": "M3@S6",
 "index_version": "corpus-5719c6dc76.parser-a53367.emb-none",
 "filters": "stations=['2'], types=['pump'], tags=['P-101']",
 "retrieved": [
  {
   "chunk_id": "R-12-r3-table3",
   "doc_id": "R-12",
   "revision": 3,
   "amendment": null,
   "clause": "А",
   "table_no": "А.1",
   "row_key": null,
   "page": 6,
   "effective_from": "2025-03-14",
   "effective_to": null
  },
  {
   "chunk_id": "R-12-r3-table1",
   "doc_id": "R-12",
   "revision": 3,
   "amendment": null,
   "clause": "4.3",
   "table_no": "3",
   "row_key": null,
   "page": 4,
   "effective_from": "2025-03-14",
   "effective_to": null
  },
  {
   "chunk_id": "R-12-r3-6",
   "doc_id": "R-12",
   "revision": 3,
   "amendment": null,
   "clause": "6",
   "table_no": null,
   "row_key": null

The retrieval set is a regression test: the gate of practice 2 (not significantly worse, at most 2 questions
broken) on S6 against S5. Topic 6 puts this into CI and runs it after every new revision or index version.

In [35]:
g = reval.gate(scores["S5"], scores["S6"])
print(f"S6 vs S5: {g['A right']} -> {g['B right']} of {g['n']}, fixed {g['B fixed']}, broke {g['B broke']} "
      f"{g['broken_cases']}, p = {g['p_value']}  ->  GATE {'PASS' if g['passed'] else 'FAIL'} {g['reasons']}")

S6 vs S5: 123 -> 130 of 143, fixed 8, broke 1 ['k02'], p = 0.0391  ->  GATE PASS []


**Holdout: are the fixes fitted to our own slices?** The retrieval set was written together with the fixes. The
holdout has 27 questions written by a model (gpt-oss-120b on the course server) from the documents and four role
descriptions only: an operator, a mechanic, a reliability engineer, an auditor. It never saw the ladder or the
slices. Code kept the gradable ones (the review list: `reference/holdout_review.md`).

In [36]:
holdout = load_cases(REPLAY_DIR.parent / "holdout.jsonl")
h_scores = {}
for name in [c.name for c in LADDER] + ["M3@S0", "M3@S6"]:
    h_scores[name] = reval.score(holdout, system.run(holdout, CONFIGS[name], progress=False), by_id)
reval.ablation(h_scores, previous)[["config", "vs", "recall@5", "evidence recall", "end-to-end", "e2e: fixed / broke",
                                    "e2e verdict"]].fillna("")

,config,vs,recall@5,evidence recall,end-to-end,e2e: fixed / broke,e2e verdict
0,S0,,13/23 = 57% [37%–74%],10/23 = 43% [26%–63%],15/27 = 56% [37%–72%],,
1,S1,S0,13/23 = 57% [37%–74%],10/23 = 43% [26%–63%],15/27 = 56% [37%–72%],+0 / -0,no detectable difference (p = 1.000)
2,S2,S1,15/23 = 65% [45%–81%],10/23 = 43% [26%–63%],15/27 = 56% [37%–72%],+1 / -1,no detectable difference (p = 1.000)
3,S2b,S1,12/23 = 52% [33%–71%],9/23 = 39% [22%–59%],13/27 = 48% [31%–66%],+0 / -2,no detectable difference (p = 0.500)
4,S3,S2b,15/23 = 65% [45%–81%],12/23 = 52% [33%–71%],20/27 = 74% [55%–87%],+8 / -1,B is better (p = 0.039)
5,S4,S3,17/23 = 74% [54%–87%],17/23 = 74% [54%–87%],19/27 = 70% [52%–84%],+0 / -1,no detectable difference (p = 1.000)
6,S5,S4,17/23 = 74% [54%–87%],18/23 = 78% [58%–90%],20/27 = 74% [55%–87%],+1 / -0,no detectable difference (p = 1.000)
7,S6,S5,17/23 = 74% [54%–87%],18/23 = 78% [58%–90%],21/27 = 78% [59%–89%],+1 / -0,no detectable difference (p = 1.000)
8,M3@S0,S0,20/23 = 87% [68%–95%],20/23 = 87% [68%–95%],23/27 = 85% [68%–94%],+8 / -0,B is better (p = 0.008)
9,M3@S6,S6,19/23 = 83% [63%–93%],20/23 = 87% [68%–95%],23/27 = 85% [68%–94%],+2 / -0,no detectable difference (p = 0.500)


On the holdout the plant-side ladder still helps (S3 is the big step), but the reranker alone on raw BM25 (M3@S0)
is at least as good as S6, and 27 questions cannot tell them apart. The honest reading: plant data are what fix versions,
applicability and conflicts, and those questions are rare in a sample of everyday questions; a reranker fixes
wording. Measure both on your own questions, and look at the cost columns before you choose.

## 9. Check yourself and reproduce

**Your table against the reference.** Retrieval columns must match exactly (BM25 and filters are deterministic);
end-to-end from replay matches too, a live run may differ by a few questions (look at the intervals).

In [37]:
mine = reval.ablation(scores, previous).set_index("config")
ref = reference.set_index("config")
common = [c for c in mine.index if c in ref.index]
pd.DataFrame({"recall@5 (yours)": mine.loc[common, "recall@5"], "recall@5 (reference)": ref.loc[common, "recall@5"],
              "end-to-end (yours)": mine.loc[common, "end-to-end"],
              "end-to-end (reference)": ref.loc[common, "end-to-end"]})

,recall@5 (yours),recall@5 (reference),end-to-end (yours),end-to-end (reference)
config,,,,
S0,61/123 = 50% [41%–58%],61/123 = 50% [41%–58%],73/143 = 51% [43%–59%],73/143 = 51% [43%–59%]
S1,65/123 = 53% [44%–61%],65/123 = 53% [44%–61%],78/143 = 55% [46%–62%],78/143 = 55% [46%–62%]
S2,69/123 = 56% [47%–65%],69/123 = 56% [47%–65%],77/143 = 54% [46%–62%],77/143 = 54% [46%–62%]
S2b,67/123 = 54% [46%–63%],67/123 = 54% [46%–63%],80/143 = 56% [48%–64%],80/143 = 56% [48%–64%]
S3,82/123 = 67% [58%–74%],82/123 = 67% [58%–74%],102/143 = 71% [63%–78%],102/143 = 71% [63%–78%]
S4,93/123 = 76% [67%–82%],93/123 = 76% [67%–82%],109/143 = 76% [69%–82%],109/143 = 76% [69%–82%]
S5,93/123 = 76% [67%–82%],93/123 = 76% [67%–82%],123/143 = 86% [79%–91%],123/143 = 86% [79%–91%]
S6,93/123 = 76% [67%–82%],93/123 = 76% [67%–82%],130/143 = 91% [85%–95%],130/143 = 91% [85%–95%]
M1t,9/18 = 50% [29%–71%],9/18 = 50% [29%–71%],13/20 = 65% [43%–82%],13/20 = 65% [43%–82%]


**Success criteria**

- the retrieval columns of your ablation table equal `reference/ablation.csv`;
- the P-101 question at M3@S6: "not normal, inspect within 24 hours", citations R-12 rev. 3 cl. 4.3.3, Table 3,
  cl. 6.1-6.2, all three checks pass (at S6 with BM25 alone it is lost in search: see step 7);
- V-912: `not_in_documents`; P-205 and the lubrication interval of P-101: `conflict` with both documents cited.

**What you built:** a retrieval set with evidence from the structure of the documents, loss attribution by link,
tag normalization, table rows, applicability through the registry, a date filter that understands amendments,
an evidence pack with references in both directions, an answer contract with three citation checks, an audit
record with the index version, and the regression gate. In the demo project this becomes the search tool of the
agent (topic 5), the regression test of the index (topic 6) and the audit log (topics 6 and 7).

**Typical mistakes** (check before you trust a number):
- the revision filter after search: the top-k is full of the old revision, after the filter nothing is left;
- `status = in force` instead of the dates: a wrong answer to a question about an incident;
- amendment No. 1 not applied: clause 4.4.1 of rev. 3 looks valid;
- the tag normalized in the chunks but not in the question (or the other way round);
- the date filter on for a "what changed" question: rev. 2 disappears;
- "dense is better than BM25" from two averages without a paired comparison; a percentage on a slice of 4;
- a citation check that only looks for the clause: the neighbouring clause 4.3.4 passes;
- `question_en` sent to search instead of `question_ru`: BM25 finds only tags and numbers;
- a new embedding profile with the old vectors: the manifest (`index_version`) must catch it;
- the course model in thinking mode spent the token limit before the JSON of the contract.

**Optional tasks** (see the README for details):
1. Reading against search: S6 on a store read from the scan by Docling (practice 3). How many losses move to "reading"?
2. Your own embedder: `bge-m3` against `qwen3-embedding:0.6b` via Ollama, paired comparison on the `language` slice.
3. LLM extraction of the query fields (`query.extract_llm`) against the rules: where do the rules miss a date or a tag?
4. Context budget: the evidence pack at 500 / 1000 / 2000 tokens: evidence recall and end-to-end.
5. Near-duplicates without metadata: remove `scope` and count the P-310 questions answered from R-12.
6. Document hierarchy: a company standard above R-12 and the rule "the stricter wins" in code; the `conflict` slice.
7. A vector database: the index in pgvector or Qdrant (installed from PyPI or a mirror); time and result.
8. Your documents: the process on your project documents: 20 questions from the structure, S0 and one ladder row.